In [1]:
! pip install torchreid

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.7/92.7 kB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for torchreid: filename=torchreid-0.2.5-py3-none-any.whl size=144324 sha256=d157d251be589a6f0fb1ce52c9ac04b499966ff5323cde54dfe91c40cec586c2
  Stored in directory: /root/.cache/pip/wheels/5c/86/ff/80a1b78a90df470cbb12c075bf189ad33f1a41a881cf9e9a09
Successfully built torchreid


In [3]:
! unzip data/archive.zip -d data/market1501

Выходные данные были обрезаны до нескольких последних строк (5000).
  inflating: data/market1501/Market-1501-v15.09.15/gt_query/1180_c2s2_164677_00_good.mat  
  inflating: data/market1501/Market-1501-v15.09.15/gt_query/1180_c2s2_164677_00_junk.mat  
  inflating: data/market1501/Market-1501-v15.09.15/gt_query/1180_c3s3_006062_00_good.mat  
  inflating: data/market1501/Market-1501-v15.09.15/gt_query/1180_c3s3_006062_00_junk.mat  
  inflating: data/market1501/Market-1501-v15.09.15/gt_query/1180_c4s5_038360_00_good.mat  
  inflating: data/market1501/Market-1501-v15.09.15/gt_query/1180_c4s5_038360_00_junk.mat  
  inflating: data/market1501/Market-1501-v15.09.15/gt_query/1180_c5s3_006593_00_good.mat  
  inflating: data/market1501/Market-1501-v15.09.15/gt_query/1180_c5s3_006593_00_junk.mat  
  inflating: data/market1501/Market-1501-v15.09.15/gt_query/1180_c6s3_029892_00_good.mat  
  inflating: data/market1501/Market-1501-v15.09.15/gt_query/1180_c6s3_029892_00_junk.mat  
  inflating: data/mark

In [4]:
import torchreid

/usr/local/lib/python3.12/dist-packages/torchreid/reid/metrics/rank.py:11: UserWarning: Cython evaluation (very fast so highly recommended) is unavailable, now use python evaluation.
  warnings.warn(


In [5]:
import torch
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [6]:
datamanager = torchreid.data.ImageDataManager(
    root="data",
    sources="market1501",
    targets="market1501",
    height=256,
    width=128,
    batch_size_train=32,
    batch_size_test=100,
    transforms=["random_flip", "random_crop", "random_erase"]
)

Building train transforms ...
+ resize to 256x128
+ random flip
+ random crop (enlarge to 288x144 and crop 256x128)
+ to torch tensor of range [0, 1]
+ normalization (mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
+ random erase
Building test transforms ...
+ resize to 256x128
+ to torch tensor of range [0, 1]
+ normalization (mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
=> Loading train (source) dataset
=> Loaded Market1501
  ----------------------------------------
  subset   | # ids | # images | # cameras
  ----------------------------------------
  train    |   751 |    12936 |         6
  query    |   750 |     3368 |         6
  gallery  |   751 |    15913 |         6
  ----------------------------------------
=> Loading test (target) dataset


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


=> Loaded Market1501
  ----------------------------------------
  subset   | # ids | # images | # cameras
  ----------------------------------------
  train    |   751 |    12936 |         6
  query    |   750 |     3368 |         6
  gallery  |   751 |    15913 |         6
  ----------------------------------------


  **************** Summary ****************
  source            : ['market1501']
  # source datasets : 1
  # source ids      : 751
  # source images   : 12936
  # source cameras  : 6
  target            : ['market1501']
  *****************************************




In [7]:
model = torchreid.models.build_model(
    name="resnet50",
    num_classes=datamanager.num_train_pids,
    loss="softmax",
    pretrained=True
)

model = model.cuda()

Downloading: "https://download.pytorch.org/models/resnet50-19c8e357.pth" to /root/.cache/torch/hub/checkpoints/resnet50-19c8e357.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 276MB/s]


In [8]:
optimizer = torchreid.optim.build_optimizer(
    model,
    optim="adam",
    lr=0.0003
)

scheduler = torchreid.optim.build_lr_scheduler(
    optimizer,
    lr_scheduler="single_step",
    stepsize=20
)

In [9]:
engine = torchreid.engine.ImageSoftmaxEngine(
    datamanager,
    model,
    optimizer=optimizer,
    scheduler=scheduler,
    label_smooth=True
)

In [ ]:
engine.run(
    save_dir="log/market1501_baseline",
    max_epoch=60,
    eval_freq=10,
    print_freq=10,
    test_only=False
)

=> Start training


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


epoch: [1/60][10/404]	time 0.183 (0.454)	data 0.000 (0.105)	eta 3:03:14	loss 7.0524 (6.8018)	acc 0.0000 (0.0000)	lr 0.000300
epoch: [1/60][20/404]	time 0.184 (0.320)	data 0.001 (0.053)	eta 2:09:02	loss 7.0689 (6.9125)	acc 0.0000 (0.1562)	lr 0.000300
epoch: [1/60][30/404]	time 0.181 (0.274)	data 0.000 (0.036)	eta 1:50:28	loss 6.8483 (6.8829)	acc 0.0000 (0.4167)	lr 0.000300
epoch: [1/60][40/404]	time 0.183 (0.251)	data 0.001 (0.027)	eta 1:41:10	loss 6.5326 (6.8216)	acc 0.0000 (0.3125)	lr 0.000300
epoch: [1/60][50/404]	time 0.183 (0.237)	data 0.001 (0.022)	eta 1:35:33	loss 6.4850 (6.7853)	acc 0.0000 (0.3750)	lr 0.000300
epoch: [1/60][60/404]	time 0.181 (0.228)	data 0.000 (0.018)	eta 1:31:50	loss 6.7622 (6.7582)	acc 0.0000 (0.3646)	lr 0.000300
epoch: [1/60][70/404]	time 0.183 (0.221)	data 0.000 (0.016)	eta 1:29:10	loss 6.6270 (6.7328)	acc 0.0000 (0.4018)	lr 0.000300
epoch: [1/60][80/404]	time 0.185 (0.217)	data 0.000 (0.014)	eta 1:27:11	loss 6.4669 (6.7170)	acc 3.1250 (0.5469)	lr 0.000300


In [ ]:
! zip -r models.zip log/market1501_baseline/model

  adding: log/market1501_baseline/model/ (stored 0%)
  adding: log/market1501_baseline/model/model.pth.tar-30 (deflated 19%)
  adding: log/market1501_baseline/model/model.pth.tar-60 (deflated 30%)
  adding: log/market1501_baseline/model/model.pth.tar-40 (deflated 24%)
  adding: log/market1501_baseline/model/model.pth.tar-10 (deflated 5%)
  adding: log/market1501_baseline/model/model.pth.tar-50 (deflated 28%)
  adding: log/market1501_baseline/model/model.pth.tar-20 (deflated 9%)


In [ ]:
from google.colab import files
files.download('models.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Загрузка весов обученной модели

In [11]:
model_test = torchreid.models.build_model(
    name="resnet50",
    num_classes=datamanager.num_train_pids,
    loss="softmax",
    pretrained=False
)

In [13]:
checkpoint = torch.load(
    'model/model.pth.tar-60',
    map_location='cpu',
    weights_only=False
)

In [14]:
state_dict = checkpoint['state_dict']
from collections import OrderedDict
new_state_dict = OrderedDict()
for k, v in state_dict.items():
    name = k[7:] if k.startswith('module.') else k
    new_state_dict[name] = v
state_dict

In [15]:
model_test.load_state_dict(new_state_dict)
model_test = model_test.cuda()

In [19]:
engine_test = torchreid.engine.ImageSoftmaxEngine(datamanager, model_test, optimizer)
engine_test.run(save_dir="log/baseline_test", test_only=True, visrank=True, visrank_topk=10)

##### Evaluating market1501 (source) #####
Extracting features from query set ...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Done, obtained 3368-by-2048 matrix
Extracting features from gallery set ...
Done, obtained 15913-by-2048 matrix
Speed: 0.0262 sec/batch
Computing distance matrix with metric=euclidean ...
Computing CMC and mAP ...
** Results **
mAP: 71.9%
CMC curve
Rank-1  : 87.8%
Rank-5  : 94.8%
Rank-10 : 96.6%
Rank-20 : 98.0%
# query: 3368
# gallery 15913
Visualizing top-10 ranks ...
- done 100/3368
- done 200/3368
- done 300/3368
- done 400/3368
- done 500/3368
- done 600/3368
- done 700/3368
- done 800/3368
- done 900/3368
- done 1000/3368
- done 1100/3368
- done 1200/3368
- done 1300/3368
- done 1400/3368
- done 1500/3368
- done 1600/3368
- done 1700/3368
- done 1800/3368
- done 1900/3368
- done 2000/3368
- done 2100/3368
- done 2200/3368
- done 2300/3368
- done 2400/3368
- done 2500/3368
- done 2600/3368
- done 2700/3368
- done 2800/3368
- done 2900/3368
- done 3000/3368
- done 3100/3368
- done 3200/3368
- done 3300/3368
Done. Images have been saved to "log/baseline_test/visrank_market1501" ...
